In [1]:
from google.colab import userdata; import os; os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

In [2]:
!pip install -q langchain-text-splitters sentence-transformers transformers torch pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 18.6 MB/s eta 0:00:00


In [3]:
import sys; sys.path.append('/content')

In [4]:
from rag_pipeline import get_or_create_index, handle_user_query

In [5]:
index = get_or_create_index('rag-multiuser-demo')

In [7]:
result = handle_user_query(index, 'user_1', 'Which rocket is the most powerful?')

[shared] document already ingested — skipping re-ingest


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
result = handle_user_query(index, 'user_1', 'Which rocket is the most powerful?')

for i, qa in enumerate(result['qa_pairs'], 1):
    print(f"[{i}] Q: {qa['question']}")
    print(f"    A: {qa['answer']}")
    print(f"    Score: {qa['rerank_score']:.4f}\n")

print("BEST ANSWER:", result['best_answer'])

[shared] document already ingested — skipping re-ingest
[1] Q: What is the largest rocket in terms of thrust?
    A: The Saturn V rocket, used in the Apollo program, remains the most powerful rocket ever successfully flown, generating 7.5 million pounds of thrust at liftoff.
    Score: 10.1354

[2] Q: Among rockets, which has the highest power output?
    A: The Saturn V rocket, used in the Apollo program, remains the most powerful rocket ever successfully flown, generating 7.5 million pounds of thrust at liftoff.
    Score: 10.1354

[3] Q: In comparison to other launch vehicles, what stands out as the strongest?
    A: The Saturn V rocket, used in the Apollo program, remains the most powerful rocket ever successfully flown, generating 7.5 million pounds of thrust at liftoff.
    Score: 10.1354

[4] Q: Which spacecraft engine produces the greatest amount of force?
    A: The Saturn V rocket generates the greatest amount of force at liftoff with 7.5 million pounds of thrust.
    Score: 

In [9]:
import rag_pipeline

def full_pipeline_fixed(index, question, user_id, top_k_per_query=5):
    variations = rag_pipeline.generate_query_variations(question)
    retrieved_lists = rag_pipeline._retrieve_parallel(index, variations, user_id, top_k_per_query)

    qa_pairs = []
    for q, retrieved in zip(variations, retrieved_lists):
        if not retrieved:
            qa_pairs.append({"question": q, "answer": "I don't have enough information to answer that.", "context": [], "faithfulness": 1.0})
            continue
        reranked_chunks = rag_pipeline.rerank(q, retrieved, top_n=3)
        context = [r["text"] for r in reranked_chunks]
        answer = rag_pipeline.generate_answer(q, context)
        faithfulness = rag_pipeline.estimate_faithfulness(answer, context)
        qa_pairs.append({"question": q, "answer": answer, "context": context, "faithfulness": round(faithfulness, 3)})

    reranker = rag_pipeline.get_reranker()
    scores = reranker.predict([[question, qa["answer"]] for qa in qa_pairs])
    for qa, s in zip(qa_pairs, scores):
        qa["rerank_score"] = float(s)

    FAITHFULNESS_THRESHOLD = 0.55
    faithful = [qa for qa in qa_pairs if qa["faithfulness"] >= FAITHFULNESS_THRESHOLD]
    unfaithful = [qa for qa in qa_pairs if qa["faithfulness"] < FAITHFULNESS_THRESHOLD]
    faithful_sorted = sorted(faithful, key=lambda x: x["rerank_score"], reverse=True)
    unfaithful_sorted = sorted(unfaithful, key=lambda x: x["rerank_score"], reverse=True)
    ranked_qa_pairs = faithful_sorted + unfaithful_sorted

    return {
        "question": question,
        "qa_pairs": ranked_qa_pairs,
        "best_answer": ranked_qa_pairs[0]["answer"],
    }

def handle_user_query_fixed(index, user_id, question, doc_filepath="corpus.txt"):
    rag_pipeline.ingest_shared_document(index, doc_filepath)
    return full_pipeline_fixed(index, question, user_id)

# Overwrite the module's functions in-place, and re-bind your local name
rag_pipeline.full_pipeline = full_pipeline_fixed
rag_pipeline.handle_user_query = handle_user_query_fixed
handle_user_query = handle_user_query_fixed

In [10]:
result = handle_user_query(index, "user_2", "How do astronauts survive re-entry?")
for i, qa in enumerate(result["qa_pairs"], 1):
    print(f"[{i}] {qa['answer']}")
    print(f"    Faithfulness: {qa['faithfulness']:.3f}\n")
print("BEST ANSWER:", result["best_answer"])

[shared] document already ingested — skipping re-ingest
[1] To optimize materials used in spacesuits and spacecraft components for extreme temperatures encountered during re-entry, engineers utilize various techniques such as:

1. **Ablative Materials**: These materials shed material through friction with the hot re-entry atmosphere, absorbing and dissipating heat before reaching critical temperature thresholds. Examples include carbon-carbon composites and ceramic tiles.

2. **Thermal Protection Surfaces (TPS)**: These surfaces are designed to withstand high temperatures without significant degradation. They can be made of materials like aluminum oxide ceramics, silicon carbide, or other advanced composite materials.

3. **Radiation Shielding**: For spacecraft components exposed to solar radiation, shielding materials are chosen to protect against harmful particle radiation. Common choices include boron nitride nanot
    Faithfulness: 0.592

[2] I don't have enough information to answ

In [11]:
import importlib
import evaluate
importlib.reload(evaluate)  # ensures it picks up the patched functions, even if you imported evaluate earlier
from evaluate import run_evaluation

run_evaluation(index)

[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest
[shared] document already ingested — skipping re-ingest

EVALUATION RESULTS: 8/8 passed (avg similarity: 0.861, threshold: 0.6)

[PASS] (0.897) Which rocket is the most powerful?

[PASS] (0.898) What year did humans first land on the Moon?

[PASS] (0.777) How fast must a rocket travel to escape Earth's gravity?

[PASS] (0.896) What is the Karman line?

[PASS] (0.717) How far is the James Webb Space Telescope from Earth?

[PASS] (0.878) What altitude does the International Space Station orbit at?

[PASS] (0.890) What company pioneered reusable rocket boosters?

[PASS] (0.932) What program aims to retur

[{'question': 'Which rocket is the most powerful?',
  'ground_truth': 'The Saturn V rocket is the most powerful rocket ever successfully flown.',
  'answer': 'The Saturn V rocket, used in the Apollo program, remains the most powerful rocket ever successfully flown, generating 7.5 million pounds of thrust at liftoff.',
  'similarity': 0.897,
  'passed': True},
 {'question': 'What year did humans first land on the Moon?',
  'ground_truth': 'Humans first landed on the Moon in 1969, during the Apollo 11 mission.',
  'answer': 'The Apollo 11 mission in 1969 was the first crewed mission to land humans on the Moon.',
  'similarity': 0.898,
  'passed': True},
 {'question': "How fast must a rocket travel to escape Earth's gravity?",
  'ground_truth': "A rocket must reach escape velocity, about 11.2 km/s, to escape Earth's gravity.",
  'answer': "To overcome Earth's gravitational influence and reach outer space, a spacecraft must accelerate to achieve escape velocity, which is approximately 11.2